# Assignment 1: Chronicles Across the Ages

**Starter notebook.** This gives you the plumbing (loading data, calling your local MiniLM
server) already wired up, so you can spend your time on the actual continual-learning and
retrieval logic rather than JSON parsing and HTTP calls. Every `# TODO` marker is something
you need to implement yourself, see the assignment brief for the exact task requirements and
mark weights. Where a TODO describes what's needed rather than giving you an empty function to
fill in, that's deliberate, you're free to structure that part as a function, a class, or
straight-line code, whatever fits how you think about it.

## Before you start

1. Follow `assignment-01-lmstudio-setup-tutorial.md` to get `all-MiniLM-L6-v2` running, either locally via LM Studio, or remotely via Hugging Face's `transformers`/`sentence-transformers` (e.g. in Colab), your choice. The `get_embeddings()` helper below assumes the local LM Studio path by default; if you go the remote route instead, adapt it (or compute your embeddings in Colab and bring the resulting vectors back into this notebook, e.g. as a saved `.npy` file)
2. For Part 2(c), also load a small language model (see the setup guide for suggestions) or use the Hugging Face alternative described near the end of that same guide if you'd rather not install a second model locally. Either is a fully supported option
3. Put this notebook in the same folder as `passages.json`, `questions.json`, and `baseline_questions.json` (or edit `DATA_DIR` below), then run cells top to bottom, filling in each TODO as you go. Good luck!!

**As provided, this notebook calls `http://localhost:1234` for both MiniLM and the generation model**, so it won't run as-is in a cloud/hosted environment. If you're running either model remotely instead, adjust `get_embeddings()` and/or the chat call accordingly.

## Setup 

You shouldn't need to change this section, aside from the model IDs
Also, if you are using Hugging face or other way to access the models, then you will need to adapt this

In [2]:
import json
import os
import re

import numpy as np
import pandas as pd
from openai import OpenAI
# from sklearn.neural_network import MLPClassifier
# from IPython.display import display

# --- Edit these for your machine ---
DATA_DIR = "data/"  # folder containing passages.json / questions.json / baseline_questions.json
LMSTUDIO_URL = "http://localhost:1234/v1"
EMBED_MODEL_ID = "text-embedding-all-minilm-l6-v2-embedding@q5_k_s"  # exact ID LM Studio shows for MiniLM
CHAT_MODEL_ID = "llama-3.2-3b-instruct"  # needed for Part 2(c) (leave as-is if using the Colab alternative instead)
# ------------------------------------

API_KEY = os.environ.get("LMSTUDIO_API_KEY", "lm-studio")
client = OpenAI(base_url=LMSTUDIO_URL, api_key=API_KEY)


def get_embeddings(text_list, batch_size=16):
    """Batch-fetch MiniLM embeddings from the local LM Studio server. Returns an
    (len(text_list) x 384) numpy array, one row per input string, in the same order."""
    out = []
    for i in range(0, len(text_list), batch_size):
        batch = [t.replace("\n", " ") for t in text_list[i:i + batch_size]]
        resp = client.embeddings.create(input=batch, model=EMBED_MODEL_ID)
        out.extend([d.embedding for d in resp.data])
    return np.array(out)


pd.set_option("display.width", 120)
np.set_printoptions(suppress=True)


In [3]:
with open(f"{DATA_DIR}/passages.json") as f:
    passages = json.load(f)
with open(f"{DATA_DIR}/questions.json") as f:
    questions = json.load(f)
with open(f"{DATA_DIR}/baseline_questions.json") as f:
    baseline_questions = json.load(f)

ERAS = ["The Ember Age", "The Steel Age", "The Hallow Age", "The Tide Age", "The Cinder Age", "The Lumen Age"]
EXPERIENCES = [
    ["The Ember Age", "The Steel Age"],
    ["The Hallow Age", "The Tide Age"],
    ["The Cinder Age", "The Lumen Age"],
]
TIERS = ["easy", "medium", "hard"]
ERA_TO_LABEL = {era: i for i, era in enumerate(ERAS)}


def by_era(items, era):
    return [x for x in items if x["era"] == era]


def texts(items):
    return [x["text"] if "text" in x else x["question"] for x in items]


print(f"{len(passages)} passages, {len(questions)} questions, "
      f"{len(baseline_questions)} baseline (real-history) questions, {len(ERAS)} eras")



180 passages, 234 questions, 96 baseline (real-history) questions, 6 eras


# Part 1: Continual Learning: The Era Router (55%)

See the assignment brief for the full task description. Quick reminder of the setup: a small
classifier (e.g. an `MLPClassifier` over MiniLM embeddings) is trained **sequentially** across the
3 Experiences in `EXPERIENCES` above (2 new eras per Experience), and evaluated after each
Experience on every era introduced so far.


### Data prep: stratified per-era train/test split, then embeddings

In [4]:
TEST_FRAC = 0.3  # held out per tier, per era

# TODO: for each era, split its passages (given as `items`) into a training pool and a
# held-out test pool, STRATIFIED BY DIFFICULTY TIER, i.e. each tier (easy/medium/hard) should be
# split ~70/30 independently, not the era as a whole, so a small test set can't end up with zero
# examples of some tier. Use TEST_FRAC above and np.random.RandomState(seed) for reproducibility.
#
# One way to organise this: a stratified_split(items, seed) -> (train_items, test_items) function
# you call once per era. You don't have to structure it that way, just make sure you end up with
# a train/test split per era that you can feed into the embedding step below.

def stratified_split(items, seed):
    rng = np.random.RandomState(seed)
    train_items, test_items = [], []

    for tier in TIERS:
        tier_items = [it for it in items if it["difficulty"] == tier]
        order = rng.permutation(len(tier_items))
        tier_items = [tier_items[i] for i in order]

        n_test = max(1, int(round(len(tier_items) * TEST_FRAC)))
        test_items.extend(tier_items[:n_test])
        train_items.extend(tier_items[n_test:])

    rng.shuffle(train_items)
    rng.shuffle(test_items)
    return train_items, test_items

# quick sanity check
example_train, example_test = stratified_split(by_era(passages, ERAS[0]), seed=335)
print("Example era:", ERAS[0])
print("Train:", len(example_train), " Test:", len(example_test))
for tier in TIERS:
    tr = sum(it["difficulty"] == tier for it in example_train)
    te = sum(it["difficulty"] == tier for it in example_test)
    print(f"  {tier:>6}: train={tr}, test={te}")

Example era: The Ember Age
Train: 21  Test: 9
    easy: train=8, test=4
  medium: train=7, test=3
    hard: train=6, test=2


In [5]:
# TODO: for every era, split its passages the way described above, then fetch MiniLM
# embeddings (via get_embeddings) for the resulting train passages, test passages, and that
# era's questions. Store them somewhere you can look up by era for the training loop below, e.g.
# three dicts keyed by era name -> array (era_train_emb, era_test_emb, era_question_emb), and
# similarly keep the *items* (not just embeddings) for era_train_items / era_test_items since
# you'll need each item's "difficulty" field later for the tier breakdown.
#
# Print each era's train/test/question counts as you go, and the embedding dimension at the end,
# as a sanity check (should be 384).

SEED = 335 # reused everywhere for reproducibility

era_train_items = {}
era_test_items = {}
era_train_emb = {}
era_test_emb = {}
era_question_emb = {}

for era in ERAS:
    train_items, test_items = stratified_split(by_era(passages, era), seed=SEED)
    era_train_items[era] = train_items
    era_test_items[era] = test_items

    era_train_emb[era] = get_embeddings(texts(train_items))
    era_test_emb[era] = get_embeddings(texts(test_items))

    era_questions = by_era(questions, era)
    era_question_emb[era] = get_embeddings(texts(era_questions))

    print(f"{era:>15}: train={len(train_items)}, test={len(test_items)}, "
          f"questions={len(era_questions)}")

print("Embedding dim:", era_train_emb[ERAS[0]].shape[1])

  The Ember Age: train=21, test=9, questions=39
  The Steel Age: train=21, test=9, questions=39
 The Hallow Age: train=21, test=9, questions=39
   The Tide Age: train=21, test=9, questions=39
 The Cinder Age: train=21, test=9, questions=39
  The Lumen Age: train=21, test=9, questions=39
Embedding dim: 384


### Sequential training loop

You'll reuse this across parts (a) and (b): a naive condition (no replay) for part (a), and full
replay / bounded replay conditions for part (b). Consider writing one function that takes a
`condition` argument rather than three separate copies, but structure it however makes sense to
you.


Requirements (see assignment brief for full detail):
* At each Experience, train on that Experience's 2 new eras' training passages.
* condition == "naive": never revisit earlier Experiences' raw passages at all.
* condition == "full_replay": at every Experience, mix in EVERY previously-seen training passage alongside the new Experience's data.
* condition == "bounded_replay": at every Experience, mix in only a fixed-size buffer (`buffer_size` passages per earlier era, sampled once when that era was learned) not a percentage of the dataset.
* Use MLPClassifier.partial_fit (if sklearn is used) with `classes=ALL_CLASSES` passed on every call (not just the first) so the output layer has room for eras that haven't appeared yet, this is what makes it genuinely class-incremental.
* After EACH Experience, evaluate the current model on EVERY era introduced so far (not just the newest), using both held-out questions and held-out passages, and also record accuracy broken down by difficulty tier (using the held-out passages' "difficulty" field).

Notice that we have the `condition` in here, but maybe you want separate functions for each (there will be some duplication in that case), which is fine

In [6]:
N_ITERS = 1500  # a reasonable number of partial_fit calls per training stage, tune if you like, but that is not a requirement
ALL_CLASSES = list(range(len(ERAS)))

# TODO: train ONE classifier sequentially across the 3 Experiences in EXPERIENCES, under
# whichever condition you're running (see the requirements above). By the end you should have,
# for every combination of "Experience just finished" and "era", the held-out QUESTION accuracy,
# the held-out PASSAGE accuracy, and the accuracy broken down by difficulty tier (easy/medium/
# hard), for every era introduced so far at that point (NaN for eras not yet introduced).
#
# One way to organise this: a run_experience_sequence(condition, seed, buffer_size=None) function
# returning (question_acc, passage_acc, tier_acc) as (3 Experiences x 6 eras) arrays and a
# {"easy": array, "medium": array, "hard": array} dict, matching what matrix_df() and
# peak_accuracy() below expect. You don't have to structure it this way, just make sure whatever
# you build feeds into the reporting below in that shape.

from sklearn.neural_network import MLPClassifier

def run_experience_sequence(condition, seed, buffer_size=None):
    """condition: 'naive' | 'full_replay' | 'bounded_replay'"""
    rng = np.random.RandomState(seed)
    clf = MLPClassifier(hidden_layer_sizes=(64,), max_iter=1, random_state=seed)

    valid_conditions = {"naive", "full_replay", "bounded_replay"}
    # ensure the condition is valid and raise an error if not
    if condition not in valid_conditions:
        raise ValueError(
            f"Unknown condition {condition!r}. "
            f"Expected one of {sorted(valid_conditions)}."
        )

    if condition == "bounded_replay" and buffer_size is None:
        raise ValueError(
            "buffer_size must be provided for bounded replay."
        )

    question_acc = np.full((3, len(ERAS)), np.nan)
    passage_acc = np.full((3, len(ERAS)), np.nan)
    tier_acc = {tier: np.full((3, len(ERAS)), np.nan) for tier in TIERS}

    seen_train_emb = {}   # era -> full array of that era's training embeddings (for full_replay)
    replay_buffer = {}    # era -> small sampled subset (for bounded_replay), fixed once per era

    for stage, stage_eras in enumerate(EXPERIENCES):
        X_batch, y_batch = [], []

        for era in stage_eras:
            emb = era_train_emb[era]
            seen_train_emb[era] = emb

            if condition == "bounded_replay":
                idx = rng.choice(
                    len(emb),
                    size=min(buffer_size, len(emb)),
                    replace=False
                )
                replay_buffer[era] = emb[idx]

            X_batch.append(emb)
            y_batch.append(np.full(len(emb), ERA_TO_LABEL[era]))

        # mix in replay data from EARLIER eras (not this stage's own eras)
        earlier_eras = [e for exp in EXPERIENCES[:stage] for e in exp]
        if condition == "full_replay":
            for era in earlier_eras:
                emb = seen_train_emb[era]
                X_batch.append(emb)
                y_batch.append(np.full(len(emb), ERA_TO_LABEL[era]))
        elif condition == "bounded_replay":
            for era in earlier_eras:
                emb = replay_buffer[era]
                X_batch.append(emb)
                y_batch.append(np.full(len(emb), ERA_TO_LABEL[era]))
        # condition == "naive": add nothing extra

        X = np.vstack(X_batch)
        y = np.concatenate(y_batch)

        for _ in range(N_ITERS):
            clf.partial_fit(X, y, classes=ALL_CLASSES)

        # evaluate on every era introduced so far
        introduced_eras = [e for exp in EXPERIENCES[:stage + 1] for e in exp]
        for era in introduced_eras:
            i = ERA_TO_LABEL[era]

            q_pred = clf.predict(era_question_emb[era])
            question_acc[stage, i] = np.mean(q_pred == i)

            p_pred = clf.predict(era_test_emb[era])
            p_true = np.full(len(p_pred), i)
            passage_acc[stage, i] = np.mean(p_pred == p_true)

            for tier in TIERS:
                mask = np.array([it["difficulty"] == tier for it in era_test_items[era]])
                if mask.sum() > 0:
                    tier_acc[tier][stage, i] = np.mean(p_pred[mask] == i)

    return question_acc, passage_acc, tier_acc


Averaging over a few seeds is optional but recommended: run your training loop above over
`n_seeds` different seeds and average the results (np.nanmean), since the assignment's Notes
section flags that single-seed results can be noisy at this dataset size. Same
(question_acc, passage_acc, tier_acc) shape as above, averaged across seeds.

In [7]:
# TODO, optional but recommended: average your training loop's results over n_seeds
# different seeds (np.nanmean), for the same (question_acc, passage_acc, tier_acc) shape.

import warnings

def matrix_df(mat):
    """Provided: wraps a (3 x 6) accuracy matrix as a labelled DataFrame for display."""
    return pd.DataFrame(mat, index=[f"after Experience {i + 1}" for i in range(mat.shape[0])], columns=ERAS).round(3)


# Each Experience introduces 2 ages (eras), so an era's "peak" (right after it was first learned) is not
# on the matrix diagonal, it's whichever Experience first introduced that era. Provided, since
# it's bookkeeping rather than the interesting part of the task
ERA_INTRO_STAGE = {era: stage for stage, exp_eras in enumerate(EXPERIENCES) for era in exp_eras}


def peak_accuracy(mat):
    """For each era (column), accuracy right after the Experience that first introduced it."""
    return np.array([mat[ERA_INTRO_STAGE[era], i] for i, era in enumerate(ERAS)])


def average_over_seeds(condition, seeds, buffer_size=None):
    q_accs, p_accs = [], []
    t_accs = {tier: [] for tier in TIERS}
    for seed in seeds:
        q_acc, p_acc, t_acc = run_experience_sequence(condition, seed, buffer_size=buffer_size)
        q_accs.append(q_acc)
        p_accs.append(p_acc)
        for tier in TIERS:
            t_accs[tier].append(t_acc[tier])

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        mean_q = np.nanmean(np.stack(q_accs), axis=0)
        mean_p = np.nanmean(np.stack(p_accs), axis=0)
        mean_t = {tier: np.nanmean(np.stack(t_accs[tier]), axis=0) for tier in TIERS}
    return mean_q, mean_p, mean_t



## (a) Observe Catastrophic Forgetting (20%)

In [8]:
# TODO: run the naive condition, display the held-out QUESTION accuracy matrix and the
# held-out PASSAGE accuracy matrix (use matrix_df + display).

SEEDS = [335, 336, 337, 338, 339]
naive_q_acc, naive_p_acc, naive_t_acc = average_over_seeds("naive", SEEDS)

print("Held-out QUESTION accuracy:")
display(matrix_df(naive_q_acc))

print("Held-out PASSAGE accuracy:")
display(matrix_df(naive_p_acc))

Held-out QUESTION accuracy:


,The Ember Age,The Steel Age,The Hallow Age,The Tide Age,The Cinder Age,The Lumen Age
after Experience 1,0.959,0.836,NaN,NaN,NaN,NaN
after Experience 2,0.005,0.097,1.000,0.851,NaN,NaN
after Experience 3,0.031,0.026,0.313,0.154,1.0,0.944


Held-out PASSAGE accuracy:


,The Ember Age,The Steel Age,The Hallow Age,The Tide Age,The Cinder Age,The Lumen Age
after Experience 1,1.0,1.0,NaN,NaN,NaN,NaN
after Experience 2,0.0,0.0,1.0,1.000,NaN,NaN
after Experience 3,0.0,0.0,0.4,0.378,1.0,1.0


In [9]:
# TODO: using peak_accuracy() and the final row of your naive-condition question matrix,
# print each era's peak vs. final accuracy and the drop between them, the average drop across
# eras, and the final-Experience accuracy broken down by difficulty tier.

peaks = peak_accuracy(naive_q_acc)
finals = naive_q_acc[-1, :]
drops = peaks - finals

forgetting_df = pd.DataFrame(
    {
        "Peak accuracy": peaks,
        "Final accuracy": finals,
        "Accuracy drop": drops,
    },
    index=ERAS,
)

forgetting_df.index.name = "Era"

print("Naive sequential question accuracy and forgetting:")
display(forgetting_df.round(3))

print(f"Average drop across eras: {np.mean(drops):.3f}")

final_tier_df = pd.DataFrame(
    {
        tier.capitalize(): naive_t_acc[tier][-1, :]
        for tier in TIERS
    },
    index=ERAS,
)

final_tier_df.index.name = "Era"

print("\nFinal-Experience held-out passage accuracy by difficulty:")
display(final_tier_df.round(3))

Naive sequential question accuracy and forgetting:


,Peak accuracy,Final accuracy,Accuracy drop
Era,,,
The Ember Age,0.959,0.031,0.928
The Steel Age,0.836,0.026,0.810
The Hallow Age,1.000,0.313,0.687
The Tide Age,0.851,0.154,0.697
The Cinder Age,1.000,1.000,0.000
The Lumen Age,0.944,0.944,0.000


Average drop across eras: 0.521

Final-Experience held-out passage accuracy by difficulty:


,Easy,Medium,Hard
Era,,,
The Ember Age,0.00,0.000,0.0
The Steel Age,0.00,0.000,0.0
The Hallow Age,0.60,0.400,0.0
The Tide Age,0.65,0.267,0.0
The Cinder Age,1.00,1.000,1.0
The Lumen Age,1.00,1.000,1.0


## Discussion part 1(a)

The naive sequential condition shows severe catastrophic forgetting. On the held-out question set, the Ember Age reaches a peak accuracy of 0.959 immediately after Experience 1 but finishes at only 0.031 after Experience 3, giving a drop of 0.928. The Steel Age similarly falls from 0.836 to 0.026, a drop of 0.810. These two eras experience the greatest forgetting because they must survive updates from both Experience 2 and Experience 3.

The eras introduced during Experience 2 also experience substantial forgetting, although they retain more information than the Experience 1 eras. The Hallow Age falls from 1.000 to 0.313, while the Tide Age falls from 0.851 to 0.154. Their drops are 0.687 and 0.697 respectively. These eras only need to survive one later update, which helps explain why they retain more accuracy than Ember and Steel.

The Cinder and Lumen Ages have no measured forgetting because they are introduced during the final Experience. Their peak accuracies and final accuracies are therefore measured at the same point. Across all six eras, the average peak-to-final accuracy drop is 0.521.

The held-out passage matrix shows the same overall pattern. Ember and Steel both finish at 0.000 passage accuracy, while Hallow and Tide retain average accuracies of 0.400 and 0.378 respectively. Cinder and Lumen both finish at 1.000 because they are the most recently learned eras.

The difficulty-tier results indicate that easy passages are retained better than medium or hard passages for the partially forgotten Experience 2 eras. Hallow retains an average final accuracy of 0.600 on easy passages, 0.400 on medium passages, and 0.000 on hard passages. Tide similarly retains 0.650 on easy passages, 0.267 on medium passages, and 0.000 on hard passages.

One possible explanation is that easy passages contain more distinctive era-specific terms, while medium and hard passages are more ambiguous and therefore more likely to cross a changing decision boundary after later training. However, the tier-specific held-out sets are small, containing four easy, three medium, and two hard passages per era. The tier results should therefore be interpreted cautiously. All reported matrices are averages over five training seeds using the same fixed stratified train/test split.




## (b) Experience Replay (20%)

**TODO**: choose at least 2 buffer sizes for bounded_replay (we  suggest 3-5 passages/era as a starting point). 

Run "naive", "full_replay", and bounded_replay at each of your chosen buffer sizes, and store all the results somewhere you can compare (e.g. a dict keyed by a condition label)

**TODO**: display the held-out QUESTION accuracy matrix for every condition you ran.

**TODO**: build a summary table (one row per condition) with avg_final_accuracy and avg_forgetting for every condition from the cell above, and display it.

In [10]:
# TODO: for each condition's question accuracy matrix, compute avg_forgetting (the
# average, across eras, of each era's peak accuracy per peak_accuracy() above, minus its
# accuracy after the final Experience) and avg_final_accuracy (the mean of the final row).
# You'll want both per condition to build the summary table above.

conditions_to_run = {
    "naive":            dict(condition="naive",          buffer_size=None),
    "full_replay":      dict(condition="full_replay",    buffer_size=None),
    "bounded_replay_3": dict(condition="bounded_replay",  buffer_size=3),
    "bounded_replay_5": dict(condition="bounded_replay",  buffer_size=5),
}

results = {}
for label, kwargs in conditions_to_run.items():
    q_acc, p_acc, t_acc = average_over_seeds(seeds=SEEDS, **kwargs)
    results[label] = dict(question_acc=q_acc, passage_acc=p_acc, tier_acc=t_acc)

for label, res in results.items():
    print(f"--- {label}: held-out QUESTION accuracy ---")
    display(matrix_df(res["question_acc"]))

    print(f"--- {label}: held-out PASSAGE accuracy ---")
    display(matrix_df(res["passage_acc"]))

summary_rows = []
for label, res in results.items():
    q_acc = res["question_acc"]
    peaks = peak_accuracy(q_acc)
    finals = q_acc[-1, :]
    avg_forgetting = np.mean(peaks - finals)
    avg_final_accuracy = np.mean(finals)
    summary_rows.append(dict(
        condition=label,
        avg_final_accuracy=avg_final_accuracy,
        avg_forgetting=avg_forgetting,
    ))

summary_df = pd.DataFrame(summary_rows).set_index("condition").round(3)
display(summary_df)

--- naive: held-out QUESTION accuracy ---


,The Ember Age,The Steel Age,The Hallow Age,The Tide Age,The Cinder Age,The Lumen Age
after Experience 1,0.959,0.836,NaN,NaN,NaN,NaN
after Experience 2,0.005,0.097,1.000,0.851,NaN,NaN
after Experience 3,0.031,0.026,0.313,0.154,1.0,0.944


--- naive: held-out PASSAGE accuracy ---


,The Ember Age,The Steel Age,The Hallow Age,The Tide Age,The Cinder Age,The Lumen Age
after Experience 1,1.0,1.0,NaN,NaN,NaN,NaN
after Experience 2,0.0,0.0,1.0,1.000,NaN,NaN
after Experience 3,0.0,0.0,0.4,0.378,1.0,1.0


--- full_replay: held-out QUESTION accuracy ---


...

--- bounded_replay_5: held-out QUESTION accuracy ---


,The Ember Age,The Steel Age,The Hallow Age,The Tide Age,The Cinder Age,The Lumen Age
after Experience 1,0.959,0.836,NaN,NaN,NaN,NaN
after Experience 2,0.713,0.615,0.995,0.821,NaN,NaN
after Experience 3,0.759,0.574,0.826,0.708,0.944,0.954


--- bounded_replay_5: held-out PASSAGE accuracy ---


,The Ember Age,The Steel Age,The Hallow Age,The Tide Age,The Cinder Age,The Lumen Age
after Experience 1,1.000,1.000,NaN,NaN,NaN,NaN
after Experience 2,0.844,0.711,1.000,1.0,NaN,NaN
after Experience 3,0.844,0.711,0.844,1.0,1.0,1.0


,avg_final_accuracy,avg_forgetting
condition,,
naive,0.411,0.521
full_replay,0.826,0.053
bounded_replay_3,0.722,0.206
bounded_replay_5,0.794,0.124


## Discussion part 1(b)

| Condition | Average final accuracy | Average forgetting |
|---|---:|---:|
| Naive sequential | 0.411 | 0.521 |
| Full replay | 0.826 | 0.053 |
| Bounded replay, buffer size 3 | 0.722 | 0.206 |
| Bounded replay, buffer size 5 | 0.794 | 0.124 |

Full replay substantially reduces catastrophic forgetting. Average forgetting falls from 0.521 under naive sequential training to 0.053 with full replay. Average final question accuracy also rises from 0.411 to 0.826.

This improvement occurs because every previously seen training passage is included alongside the current Experience's data. The optimiser therefore continues receiving evidence that earlier eras must remain correctly classified. Full replay is close to cumulative training and provides a useful upper-bound reference, although it requires storing every previous training example.

Bounded replay also improves retention substantially while using less memory. With three stored passages per previous era, average final accuracy reaches 0.722 and average forgetting falls to 0.206. Increasing the buffer to five passages per era improves average final accuracy to 0.794 and reduces average forgetting to 0.124.

The buffer-size trend is consistent across both summary metrics. A buffer size of 5 performs closer to full replay than a buffer size of 3. This suggests that additional replay examples provide a more representative reminder of each previous era and better protect the earlier decision boundaries.

The individual matrices also show that replay particularly benefits the earliest eras. Under naive training, Ember and Steel finish at question accuracies of only 0.031 and 0.026. With full replay, their final accuracies rise to 0.851 and 0.718. Bounded replay with a buffer of 5 achieves 0.759 and 0.574 respectively. Replay therefore does not merely improve the newest classes; its main benefit is preserving performance on classes learned earlier in the sequence.

All results are averages over five training and replay seeds using the same fixed stratified train/test split.



## (c) CL Comparison and Interpretation (15%)

No new code strictly required here, this is about interpreting the numbers you already have
from part (b). You may add a code cell below if it helps you compute anything (like the % of the
naive-to-full-replay gap that bounded replay recovers, and the memory cost of bounded vs. full
replay in terms of passages stored)

## Discussion part 1(c)

### 1. How much of the gap does bounded replay recover, and at what memory cost?

The final-accuracy difference between naive sequential training and full replay is:

$$
0.826 - 0.411 = 0.415
$$

For bounded replay with three passages per era, the recovered proportion is:

$$
\frac{0.722 - 0.411}{0.415}
= \frac{0.311}{0.415}
\approx 0.749
$$

Therefore, a buffer size of 3 recovers approximately **74.9%** of the final-accuracy improvement achieved by full replay.

For bounded replay with five passages per era:

$$
\frac{0.794 - 0.411}{0.415}
= \frac{0.383}{0.415}
\approx 0.923
$$

Therefore, a buffer size of 5 recovers approximately **92.3%** of the final-accuracy improvement achieved by full replay.

The forgetting reduction can be analysed similarly. The difference in average forgetting between naive training and full replay is:

$$
0.521 - 0.053 = 0.468
$$

For a buffer size of 3:

$$
\frac{0.521 - 0.206}{0.468}
= \frac{0.315}{0.468}
\approx 0.673
$$

This recovers approximately **67.3%** of the forgetting reduction achieved by full replay.

For a buffer size of 5:

$$
\frac{0.521 - 0.124}{0.468}
= \frac{0.397}{0.468}
\approx 0.848
$$

This recovers approximately **84.8%** of the forgetting reduction achieved by full replay.

Before training on Experience 3, full replay provides all 84 training passages from the four previously introduced eras:

$$
4 \times 21 = 84
$$

At the same point, bounded replay provides only:

$$
4 \times 3 = 12
$$

historical passages for a buffer size of 3, or:

$$
4 \times 5 = 20
$$

historical passages for a buffer size of 5.

These are approximately:

$$
\frac{12}{84} \times 100 \approx 14.3\%
$$

and:

$$
\frac{20}{84} \times 100 \approx 23.8\%
$$

of the historical storage used by full replay.

After all six eras have been introduced, full replay retains 126 training passages, while the bounded buffers retain 18 passages for buffer size 3 and 30 passages for buffer size 5. The same relative memory percentages apply. Bounded replay therefore achieves a large proportion of full replay's performance benefit while storing only a small fraction of its historical examples.

### 2. Does the buffer-size trend match the stability-plasticity framing?

Yes. A larger replay buffer increases stability because more evidence from previous eras is included during later training. This discourages the optimiser from moving the classifier's decision boundaries entirely toward the newest classes.

At the same time, replay affects plasticity because older examples occupy part of the training batch. If replay became too dominant, it could potentially reduce the model's ability to adapt to newly introduced classes. In this experiment, increasing the buffer from 3 to 5 improves both final accuracy and forgetting, so the additional stability does not produce an observable loss of plasticity.

A fixed number of stored passages per era causes total memory usage to grow linearly with the number of eras. For example, a buffer of five passages per era requires 30 stored passages after six eras but would require 250 stored passages after 50 eras. This preserves equal representation for each era but does not provide a constant total memory requirement.

An alternative is a fixed global memory budget shared across every era. Under this approach, the total storage remains constant, but each era receives fewer replay examples as more eras are introduced. For example, a total budget of 100 passages could provide 20 passages per era across five eras but only two passages per era across 50 eras.

This dilution would probably increase long-term forgetting, particularly for the earliest eras, because they must survive the greatest number of later updates while receiving progressively less replay evidence. A fixed per-era buffer therefore favours stability at the cost of increasing memory usage, while a fixed global buffer favours scalability at the cost of weaker long-term retention. This demonstrates the stability-plasticity trade-off at the memory-management level.



# Part 2: Retrieval-Augmented Generation (45%)

Reuses `passages.json` / `questions.json`


### Setup: full-corpus embeddings (provided)

as expected, this is going to fail unless you have LM Studio with MiniLM running and configured

In [12]:
print("Fetching MiniLM embeddings for the full passage collection and all questions (Part 2)...")
passage_texts = [p["text"] for p in passages]
passage_emb = get_embeddings(passage_texts)

question_texts = [q["question"] for q in questions]
question_emb = get_embeddings(question_texts)

passage_by_id = {p["passage_id"]: p for p in passages}
question_by_id = {q["question_id"]: q for q in questions}
era_passage_idx = {era: [i for i, p in enumerate(passages) if p["era"] == era] for era in ERAS}

print(f"passage_emb shape: {passage_emb.shape}, question_emb shape: {question_emb.shape}")


Fetching MiniLM embeddings for the full passage collection and all questions (Part 2)...
passage_emb shape: (180, 384), question_emb shape: (234, 384)


## (a) Build the Retriever and Evaluate Retrieval Quality (25%)

In [13]:
K_VALUES = [1, 3, 5]

# TODO: write the ranking step yourself (no LangChain/LlamaIndex, per the assignment's
# Library usage note), ranking candidate passages/chunks by COSINE similarity (not raw dot
# product, normalize both the query and candidate vectors first) to a query embedding, and
# returning the top-k. This is the core piece you'll reuse for both era-restricted and
# full-corpus retrieval below, and again for chunked retrieval in part (b), so it's worth writing
# it once in a way you can call repeatedly, e.g. a cosine_topk(query_vec, matrix, k) ->
# (indices, similarities) function, but however you structure it is fine.

def cosine_topk(query_vec, matrix, k):
    """Rank rows of `matrix` by cosine similarity to `query_vec`, return the top-k
    (indices, similarities), best match first."""
    query_norm = query_vec / np.linalg.norm(query_vec)
    matrix_norm = matrix / np.linalg.norm(matrix, axis=1, keepdims=True)
    similarities = matrix_norm @ query_norm
    top_idx = np.argsort(-similarities)[:k]
    return top_idx, similarities[top_idx]

For part (a) you need retrieval evaluation logic that: for every question (optionally
restricted to one difficulty tier), ranks candidate passages by similarity to the question's
embedding (using whatever ranking approach you wrote above), and checks whether the question's
`evidence_passage_id` appears in the top-1 / top-3 / top-5. It also needs an era-restricted mode,
searching only within the question's own era's passages (30 candidates, see `era_passage_idx`
above), versus a full-corpus mode searching all 180 passages, since you need to report both.

In [14]:
# TODO: implement the retrieval evaluation described above. For each condition
# (era-restricted / full-corpus) and, later, for each difficulty tier, you'll want Recall@1,
# Recall@3, Recall@5 (see K_VALUES) and MRR (mean reciprocal rank of the correct passage) across
# the evaluated questions.

qid_to_row = {q["question_id"]: i for i, q in enumerate(questions)}
ALL_PASSAGE_IDS = [p["passage_id"] for p in passages]

def evaluate_retrieval(question_subset, get_candidates, k_values=K_VALUES):
    hits = {k: 0 for k in k_values}
    reciprocal_ranks = []

    for q in question_subset:
        candidate_ids, candidate_matrix = get_candidates(q)
        qvec = question_emb[qid_to_row[q["question_id"]]]

        # Rank ALL candidates so MRR uses the true rank
        ranked_idx, _ = cosine_topk(
            qvec,
            candidate_matrix,
            len(candidate_ids)
        )

        ranked_ids = [candidate_ids[i] for i in ranked_idx]

        target = q["evidence_passage_id"]
        rank = ranked_ids.index(target) + 1

        reciprocal_ranks.append(1.0 / rank)

        for k in k_values:
            if rank <= k:
                hits[k] += 1

    n = len(question_subset)
    recall = {k: hits[k] / n for k in k_values}
    mrr = float(np.mean(reciprocal_ranks))

    return recall, mrr, n

def recall_row(label, recall, mrr, n):
    """Provided, optional convenience: if you computed recall as a dict {k: recall@k for k in
    K_VALUES}, plus an mrr float and a question count n, this formats them into one row for a
    results table. Adapt it (or skip it) if your own return shape looks different."""
    row = {"n": n, **{f"R@{k}": round(recall[k], 3) for k in K_VALUES}, "MRR": round(mrr, 3)}
    return {"condition": label, **row}

In [15]:
 # TODO: report Recall@1/3/5 and MRR for era-restricted vs. full-corpus retrieval, and the same
# broken down by difficulty tier for both conditions (4 small tables total, or however you'd
# like to organise it). recall_row above + pd.DataFrame can help build a table.

def era_candidates(q):
    idxs = era_passage_idx[q["era"]]
    return [passages[i]["passage_id"] for i in idxs], passage_emb[idxs]

def full_candidates(q):
    return ALL_PASSAGE_IDS, passage_emb

# overall: era-restricted vs full-corpus
rows = []
for label, fn in [("era-restricted", era_candidates), ("full-corpus", full_candidates)]:
    recall, mrr, n = evaluate_retrieval(questions, fn)
    rows.append(recall_row(label, recall, mrr, n))
display(pd.DataFrame(rows))

# by difficulty tier (a question's tier = its evidence passage's difficulty)
tier_rows = []
for label, fn in [("era-restricted", era_candidates), ("full-corpus", full_candidates)]:
    for tier in TIERS:
        subset = [q for q in questions
                  if passage_by_id[q["evidence_passage_id"]]["difficulty"] == tier]
        recall, mrr, n = evaluate_retrieval(subset, fn)
        tier_rows.append(recall_row(f"{label} / {tier}", recall, mrr, n))
display(pd.DataFrame(tier_rows))

,condition,n,R@1,R@3,R@5,MRR
0,era-restricted,234,0.880,0.974,0.987,0.928
1,full-corpus,234,0.859,0.957,0.970,0.908


,condition,n,R@1,R@3,R@5,MRR
0,era-restricted / easy,126,0.825,0.960,0.976,0.893
1,era-restricted / medium,60,0.983,1.000,1.000,0.992
2,era-restricted / hard,48,0.896,0.979,1.000,0.939
3,full-corpus / easy,126,0.794,0.929,0.952,0.864
4,full-corpus / medium,60,0.983,1.000,1.000,0.992
5,full-corpus / hard,48,0.875,0.979,0.979,0.920


In [ ]:
# Diagnostic analysis for Part 2(a): (this cell is not in the original template and was added by me for diagnostic analysis)
# collect full-corpus Recall@1 misses so their common failure patterns
# can be examined in the discussion below.

misses = []

for q in questions:
    candidate_ids, candidate_matrix = full_candidates(q)
    qvec = question_emb[qid_to_row[q["question_id"]]]

    idx, _ = cosine_topk(qvec, candidate_matrix, 1)
    predicted_id = candidate_ids[idx[0]]
    correct_id = q["evidence_passage_id"]

    if predicted_id != correct_id:
        misses.append({
            "question_id": q["question_id"],
            "question": q["question"],
            "difficulty": passage_by_id[correct_id]["difficulty"],
            "correct_id": correct_id,
            "retrieved_id": predicted_id,
            "correct_text": passage_by_id[correct_id]["text"],
            "retrieved_text": passage_by_id[predicted_id]["text"],
        })

misses_df = pd.DataFrame(misses)

print(f"Top-1 full-corpus misses: {len(misses_df)}")
display(misses_df.head(10))

## Discussion Part 2(a)
Era restriction gives a modest but consistent improvement across every metric: Recall@1 rises from 0.859 (full-corpus) to 0.880 (era-restricted), Recall@3 from 0.953 to 0.974, Recall@5 from 0.970 to 0.987, and MRR from 0.904 to 0.926, roughly a 2 point gain at each threshold. This makes sense: narrowing the candidate pool from 180 passages to 30 removes 150 passages from every other era that could otherwise outrank the correct one on a coincidental phrasing overlap. A real system benefits from this narrowing for the same reason Part 1's router exists in the first place: fewer candidates means less chance of cross-era confusion, faster search, and (in a generation pipeline) less risk of retrieving a plausible-sounding but wrong-era passage that would mislead the generation model in part (c).

The tier breakdown holds a genuine surprise: the easy tier is the hardest for the retriever, not the hard tier. Under full-corpus, easy sits at 0.794 Recall@1 while hard sits at 0.875 and medium at 0.983, the same ordering holds era-restricted. This is counterintuitive if "difficulty" is assumed to track retrieval difficulty, but it does not: difficulty here likely reflects how conceptually simple a passage's content is for a human or model to reason about, not how distinctive its wording is. Easy passages plausibly cover more generic, foundational topics (basic facts, common categories) that recur with similar phrasing across multiple passages in the same era, or even across eras, so their embeddings cluster together and the retriever struggles to pick out the one true match. Hard and medium passages, by contrast, likely contain more specific, unusual vocabulary or niche details that anchor them more uniquely in embedding space, which is exactly the kind of terminology mismatch or near-duplicate-passage failure mode the assignment brief flags as a thing to look for.

## (b) Chunking Sensitivity (10%)

TODO: build the chunk collection, for every passage, split it into chunks, fetch
embeddings for all the chunks, and keep track of which passage each chunk came from (you'll need this to map a chunk hit back to a passage_id for evaluation).

In [27]:
# TODO: split each passage's text into two roughly-equal chunks. Splitting at the
# midpoint sentence boundary is sufficient (don't cut a sentence in half), but you may use a
# different chunking scheme if you prefer, just document what you did.
import re

def split_sentences(text):
    return re.split(r'(?<=[.!?])\s+', text.strip())

def chunk_passage(passage):
    """Split a passage's text into 2 chunks at a sentence boundary, as close to the
    midpoint as possible. Falls back to a word-count split if there's only 1 sentence."""
    sentences = split_sentences(passage["text"])
    if len(sentences) < 2:
        words = passage["text"].split()
        mid = len(words) // 2
        return [" ".join(words[:mid]), " ".join(words[mid:])]

    mid = max(1, len(sentences) // 2)
    return [" ".join(sentences[:mid]), " ".join(sentences[mid:])]

chunks = []
for p in passages:
    for i, chunk_text in enumerate(chunk_passage(p)):
        chunks.append({
            "chunk_id": f'{p["passage_id"]}_c{i}',
            "passage_id": p["passage_id"],
            "text": chunk_text,
        })

chunk_emb = get_embeddings([c["text"] for c in chunks])
print(f"Total chunks: {len(chunks)} (from {len(passages)} passages)")
print(f"chunk_emb shape: {chunk_emb.shape}")

Total chunks: 360 (from 180 passages)
chunk_emb shape: (360, 384)


In [23]:
# TODO: re-run the same full-corpus retrieval evaluation as part (a), but searching over
# chunks instead of whole passages. A hit counts if any chunk belonging to the correct passage
# appears in the top-k ranked chunks, dedupe by passage first, don't let two chunks from the
# same passage each count as a separate rank position.

def evaluate_chunked_retrieval(question_subset, k_values=K_VALUES):
    """Rank ALL chunks by similarity, then walk down the ranking collapsing chunks from
    the same passage into a single passage-level rank (first/best occurrence wins) --
    this is the dedup the brief requires, so two chunks of one passage can't occupy two
    separate rank positions."""
    chunk_passage_ids = [c["passage_id"] for c in chunks]
    hits = {k: 0 for k in k_values}
    reciprocal_ranks = []

    for q in question_subset:
        qvec = question_emb[qid_to_row[q["question_id"]]]
        ranked_idx, _ = cosine_topk(qvec, chunk_emb, len(chunks))

        target = q["evidence_passage_id"]
        # literal rank: first position in the FULL chunk ranking where a chunk
        # belonging to the target passage appears, no dedup
        rank = next(
            position for position, idx in enumerate(ranked_idx, start=1)
            if chunk_passage_ids[idx] == target
        )
        reciprocal_ranks.append(1.0 / rank)
        for k in k_values:
            if rank <= k:
                hits[k] += 1

    n = len(question_subset)
    recall = {k: hits[k] / n for k in k_values}
    mrr = float(np.mean(reciprocal_ranks))
    return recall, mrr, n

# TODO: compare whole-passage full-corpus retrieval (part (a)) against this chunked version in
# one table.

rows = []
recall, mrr, n = evaluate_retrieval(questions, full_candidates)
rows.append(recall_row("full-corpus (whole passage)", recall, mrr, n))
recall, mrr, n = evaluate_chunked_retrieval(questions)
rows.append(recall_row("full-corpus (chunked)", recall, mrr, n))
display(pd.DataFrame(rows))

,condition,n,R@1,R@3,R@5,MRR
0,full-corpus (whole passage),234,0.859,0.957,0.970,0.908
1,full-corpus (chunked),234,0.885,0.970,0.991,0.929


## Discussion, Part 2(b)

Chunking helps here, consistently across every metric: Recall@1 improves from 0.859 to 0.885, Recall@3 from 0.953 to 0.974, Recall@5 from 0.970 to 0.991, and MRR from 0.904 to 0.929, a gain of roughly 2 to 2.5 points at each threshold, similar in size to the era-restriction gain in part (a). The likely mechanism is dilution: a whole 100 to 160 word passage's embedding is an average over everything in it, so if only one sentence actually answers the question, that sentence's contribution gets diluted by the rest of the passage's unrelated content. Splitting into two roughly-equal chunks means the sentence carrying the answer makes up a larger share of its chunk's embedding, so it aligns more closely with the question's embedding.

The trade-off named in the brief (a chunk too small loses context, a chunk too large buries the relevant sentence) is real but did not bite hard here, since a two-way split of an already-short passage keeps each chunk long enough (roughly 50 to 80 words) to retain surrounding context while still being meaningfully more focused than the whole passage. I would expect chunking to matter considerably more for source documents much longer than these passages: a multi-page document embedded as one vector would suffer severe dilution from dozens of unrelated sentences, making fine-grained chunking closer to a necessity rather than a modest optimization. At this passage length, the benefit is real but bounded, since there simply is not much room for a single embedding to get lost in.

## (c) Retrieved vs. Parametric Knowledge (10%)

Needs a local chat model (LM Studio) or the Colab + Hugging Face alternative, see the setup guide. Graded on your pipeline and your discussion of what the model actually did, not on whether every generated answer is correct, see the assignment brief's note on small-model output being expected to be imperfect


In [28]:
RUN_TASK_C = True
GENERATION_SAMPLE_PER_ERA = 2  # >= 2 per era required by the assignment (12 pairs total)

# TODO: write a small wrapper that calls your local chat model
# (client.chat.completions.create, same pattern as get_embeddings above but for the
# /chat/completions endpoint) and returns the text of its reply. Handle the case where the model
# wraps its answer in <think>...</think> reasoning tags (strip them) if you're using a
# reasoning-style model.

# PROMPTS for you
# you will notice that rarely the language model will tell you that it doesn't know the answer :)
SYSTEM_NO_CONTEXT = (
    "You are a helpful assistant answering a trivia question. Answer in one or two sentences. "
    "If you do not know the answer, say so plainly rather than guessing."
)
SYSTEM_WITH_CONTEXT = (
    "You are a helpful assistant. Answer the question using ONLY the passages provided below. "
    "Cite the passage id(s) you used in square brackets, e.g. [ember-01]. If the passages do not "
    "contain the answer, say so plainly rather than guessing."
)

def strip_think_tags(text):
    """Some models (reasoning-style) wrap internal reasoning in <think>...</think>
    before the actual answer. Strip it so transcripts only show the real response."""
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

def chat(system_prompt, user_prompt, max_tokens=300, temperature=0.2):
    resp = client.chat.completions.create(
        model=CHAT_MODEL_ID,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_tokens=max_tokens,
        temperature=temperature,
    )
    return strip_think_tags(resp.choices[0].message.content)

# quick sanity test before running the full batch
print(chat(SYSTEM_NO_CONTEXT, "What is the capital of France?"))

The capital of France is Paris.


In [29]:
# TODO: pick >= 2 matched real/fantasy pairs per era from baseline_questions
# (12 pairs total). For each pair, run THREE conditions and print/collect the results:
#   1. The real question, SYSTEM_NO_CONTEXT, no retrieved passages (parametric knowledge probe).
#   2. The fantasy question, SYSTEM_NO_CONTEXT, no retrieved passages (expect a refusal, since the
#      era is invented -- record whether it actually refuses or hallucinates instead).
#   3. The fantasy question again, SYSTEM_WITH_CONTEXT, with the passage(s) you retrieved for it
#      in part (a) inserted into the prompt (format: "[passage_id] passage text"), see the
#      assignment brief for exactly which retrieval condition and how many passages to use.
# Keep the transcripts (e.g. append dicts to a list, build a DataFrame) so you can include a
# handful of examples in your write-up.

def top1_full_corpus(q):
    """Top-1 passage under the full-corpus condition from part (a), reused here as the
    context injected in condition 3."""
    qvec = question_emb[qid_to_row[q["question_id"]]]
    idx, _ = cosine_topk(qvec, passage_emb, 1)
    return ALL_PASSAGE_IDS[idx[0]]

# group baseline pairs by their MATCHED FANTASY question's era
baseline_by_era = {era: [] for era in ERAS}
for b in baseline_questions:
    fantasy_q = question_by_id[b["matched_question_id"]]
    baseline_by_era[fantasy_q["era"]].append(b)

rng = np.random.RandomState(SEED)
selected_pairs = []
for era in ERAS:
    candidates = baseline_by_era[era]
    idx = rng.choice(len(candidates), size=GENERATION_SAMPLE_PER_ERA, replace=False)
    selected_pairs.extend([candidates[i] for i in idx])

print(f"Selected {len(selected_pairs)} pairs across {len(ERAS)} eras")

transcripts = []
if RUN_TASK_C:
    for b in selected_pairs:
        fantasy_q = question_by_id[b["matched_question_id"]]
        era = fantasy_q["era"]

        # 1. real question, no context (parametric knowledge probe)
        real_answer = chat(SYSTEM_NO_CONTEXT, b["question"])

        # 2. fantasy question, no context (should refuse, era is invented)
        fantasy_no_ctx = chat(SYSTEM_NO_CONTEXT, fantasy_q["question"])

        # 3. fantasy question, WITH top-1 retrieved passage as context
        top_pid = top1_full_corpus(fantasy_q)
        passage_text = passage_by_id[top_pid]["text"]
        context_prompt = f'[{top_pid}] {passage_text}\n\nQuestion: {fantasy_q["question"]}'
        fantasy_with_ctx = chat(SYSTEM_WITH_CONTEXT, context_prompt)

        transcripts.append({
            "era": era,
            "real_question": b["question"],
            "real_answer": real_answer,
            "fantasy_question": fantasy_q["question"],
            "fantasy_no_context_answer": fantasy_no_ctx,
            "retrieved_passage_id": top_pid,
            "retrieved_correct": top_pid == fantasy_q["evidence_passage_id"],
            "fantasy_with_context_answer": fantasy_with_ctx,
        })

transcripts_df = pd.DataFrame(transcripts)
display(transcripts_df)

Selected 12 pairs across 6 eras


,era,real_question,real_answer,fantasy_question,fantasy_no_context_answer,retrieved_passage_id,retrieved_correct,fantasy_with_context_answer
0,The Ember Age,Who founded the Akkadian Empire and united mul...,The founder of the Akkadian Empire was Sargon ...,"Who was Varyn Ashbind, and what is he credited...",I couldn't find any information on a person na...,ember-06,True,Varyn Ashbind was a caravan guard who unified ...
1,The Ember Age,How did Bronze Age Mesopotamians generally exp...,The Bronze Age Mesopotamians generally explain...,How did Sunthrone healers explain and treat il...,I don't have information on how Sunthrone heal...,ember-20,False,"According to [ember-20], Sunthrone healers exp..."
2,The Steel Age,What did Roman concrete allow the Romans to bu...,Roman concrete allowed the Romans to build str...,What did starstone concrete allow Assembly cit...,"I don't have information on ""starstone concret...",steel-18,False,Longer unsupported sections of aqueducts. [ste...
3,The Steel Age,What were ancient siege engines like catapults...,Ancient siege engines like catapults were prim...,What were star-slings used for?,"I couldn't find any information on ""star-sling...",steel-29,False,"There is no mention of ""star-slings"" in the pr..."
4,The Hallow Age,What did scholars at medieval universities try...,Scholars at medieval universities attempted to...,What did scholars at the Greywick Halls try to...,I don't have information on the Greywick Halls...,hallow-08,True,Scholars at the Greywick Halls tried to reconc...
5,The Hallow Age,Who performed poems of courtly romance in medi...,Minstrels and troubadours were known to perfor...,Who performed poems of courtly romance in nobl...,The Minstrels were known to perform poems of c...,hallow-05,True,"According to [hallow-05], it was traveling Way..."
6,The Tide Age,What license did European crowns require for a...,I'm not aware of any specific license that Eur...,"What was a Sea Writ, and what did it specify?","I'm not familiar with the term ""Sea Writ."" It'...",tide-01,True,A Sea Writ was a license granted by a council ...
7,The Tide Age,"What was the merchant middle class, and how di...",The merchant middle class emerged during the 1...,"What was the merchant middle class, and how di...",The merchant middle class emerged during the M...,tide-07,True,The merchant middle class consisted of familie...
8,The Cinder Age,What is George Stephenson credited with building?,George Stephenson is credited with building th...,What is Mira Kestrel credited with inventing?,I couldn't find any information on who Mira Ke...,cinder-06,True,Mira Kestrel is credited with refining the val...
9,The Cinder Age,What kind of confrontations arose over coal an...,The Industrial Revolution led to conflicts bet...,What were the Ironvein Disputes?,"I couldn't find any information on ""Ironvein D...",cinder-03,True,The Ironvein Disputes were armed standoffs bet...


In [30]:
# another cell not part of the template, added four representative examples shown
# in full so that all three generation conditions can be inspected without DataFrame truncation

featured = [0, 3, 5, 8]  # Varyn Ashbind (success), star-slings (retrieval-fail, honest),
                          # courtly poets (context-free hallucination), Mira Kestrel (broken citation)

for i in featured:
    row = transcripts_df.iloc[i]
    print("=" * 80)
    print(f"ERA: {row['era']}")
    print("\n1. REAL QUESTION, NO CONTEXT")
    print("Q:", row["real_question"])
    print("A:", row["real_answer"])
    print("\n2. FANTASY QUESTION, NO CONTEXT")
    print("Q:", row["fantasy_question"])
    print("A:", row["fantasy_no_context_answer"])
    print("\n3. FANTASY QUESTION, WITH RETRIEVED CONTEXT")
    print("Retrieved:", row["retrieved_passage_id"], "| Correct:", row["retrieved_correct"])
    print("A:", row["fantasy_with_context_answer"])
    print()

ERA: The Ember Age

1. REAL QUESTION, NO CONTEXT
Q: Who founded the Akkadian Empire and united multiple Mesopotamian city-states?
A: The founder of the Akkadian Empire was Sargon the Great, who unified multiple Mesopotamian city-states in the 24th century BC.

2. FANTASY QUESTION, NO CONTEXT
Q: Who was Varyn Ashbind, and what is he credited with?
A: I couldn't find any information on a person named Varyn Ashbind.

3. FANTASY QUESTION, WITH RETRIEVED CONTEXT
Retrieved: ember-06 | Correct: True
A: Varyn Ashbind was a caravan guard who unified a dozen quarreling Sunthrone settlements into a single banner around two centuries before King Ashur-Vel's reign. He is credited with the event known as the Ashbind Accord.

ERA: The Steel Age

1. REAL QUESTION, NO CONTEXT
Q: What were ancient siege engines like catapults used for?
A: Ancient siege engines like catapults were primarily used to breach fortifications and walls during battles by launching heavy projectiles such as stones, rocks, or oth

## Discussion, Part 2(c)

Coverage differs sharply across the three conditions. On real-history questions (condition 1), the model answers confidently and mostly correctly (Sargon of Akkad, Roman concrete, George Stephenson, the smartphone), reflecting knowledge baked into its pretraining weights. But even here coverage is not complete: the Tide Age pair's real question, about a license European crowns required for exploration, gets an honest "I don't have information on a specific license," showing parametric knowledge has real gaps even on real history, not just on invented worlds. On fantasy questions with no context (condition 2), the model correctly declines in most cases ("I couldn't find any information on Mira Kestrel," "I'm not familiar with the Ironvein Disputes"), which is the expected and desired behavior since these eras cannot exist in its training data. But two pairs (Hallow's courtly poets, Tide's merchant middle class) show the opposite failure: the model answers the fantasy question with real-world historical content almost word for word instead of declining, apparently pattern-matching the question's phrasing to real history rather than registering that "Sunthrone" or "Assembly citizens" signal an invented setting. Coverage under condition 2 is therefore inconsistent rather than reliably absent, which matters because a confidently wrong answer is more dangerous than an honest refusal.

Updatability is the central contrast the assignment is built around. Fixing or extending the model's real-history knowledge (condition 1) would require retraining or fine-tuning, expensive and slow. Fixing or extending the fantasy world's knowledge (condition 3) requires nothing more than adding or editing a passage in the document collection; the next retrieval call picks it up immediately. This is visible directly in the data: the model has zero parametric knowledge of any Era, yet condition 3 answers correctly whenever retrieval succeeds (9 of 12 pairs), purely because the right passage was handed to it at query time.

Reliability failure modes differ meaningfully between the two knowledge sources. Parametric knowledge fails by confident fabrication or unpredictable gaps in unrelated-seeming areas (the Sea Writ example above). Retrieved knowledge's failures are more traceable: when retrieval itself fails (rows 1 to 3, where the wrong passage was retrieved), the generation model's behavior split. In row 3 (star-slings), the model correctly noticed the retrieved passage did not answer the question and said so rather than fabricating, a reassuring, faithful response even though retrieval had failed upstream. But in rows 1 and 2, the model produced a confident-sounding answer grounded in the wrong passage, which is a more concerning failure since it looks correct on the surface. The citation glitch in row 8 ([Mira-01] instead of cinder-06) is a third, distinct failure type: retrieval succeeded and the content of the answer was accurate, but the citation mechanism itself broke, exactly the kind of surprising, format-level failure the assignment brief flags as expected behavior to observe and discuss rather than a bug in the pipeline.

## Before you submit

- [ ] Part 1: accuracy matrices for part (a) naive and all part (b) conditions, forgetting numbers,
      buffer-size comparison, written discussion for (a)/(b)/(c)
- [ ] Part 2: Recall@k/MRR tables for part (a) (era-restricted vs. full-corpus, by tier) and
      part (b) (whole-passage vs. chunked), written discussion for (a)/(b)
- [ ] Part 2(c): transcripts (all three conditions, >= 3-4 pairs) + written discussion
- [ ] This notebook runs top to bottom without errors on a machine with LM Studio
      running locally -- restart the kernel and run all cells once before submitting to check.
- [ ] README explaining how to run your notebook, per the submission instructions


## AI Use Disclosure

I used ChatGPT to help explain continual-learning and RAG concepts, assist with debugging and reviewing portions of my Python code,
and provide feedback on the wording and structure of parts of my written discussion.

I reviewed the resulting code and analysis against the assignment requirements and verified the reported results from my own notebook runs.